## 1. 환경 설정

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(".env", usecwd=True), override=True)

True

`(2) 라이브러리`

In [2]:
import re
import os, json

from textwrap import dedent
from pprint import pprint

import warnings
warnings.filterwarnings("ignore")

## 2. 도구 호출 (Tool Calling)
- 도구 호출은 LLM이 특정 작업을 수행하기 위해 외부 기능을 호출하는 기능
- 이를 통해 LLM은 외부 API 통합 등 더 복잡한 작업을 수행할 수 있음 

### 2-1. 랭체인 내장 도구
- Tavily 웹 검색 도구 (예시)

`(1) 도구(tool) 정의하기`

In [3]:
from langchain_community.tools import TavilySearchResults

# 검색할 쿼리 설정
query = "스테이크와 어울리는 와인을 추천해주세요."

# Tavily 검색 도구 초기화 (최대 2개의 결과 반환)
web_search = TavilySearchResults(max_results=2)

# 웹 검색 실행
search_results = web_search.invoke(query)

# 검색 결과 출력
for result in search_results:
    print(result)
    print("-" * 100)

/tmp/ipykernel_1261816/2946721529.py:7: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(max_results=2)


{'title': '스테이크와 어울리는 최고의 와인: 무엇을 고를 것인가? - 마시자 매거진', 'url': 'https://mashija.com/%EC%8A%A4%ED%85%8C%EC%9D%B4%ED%81%AC%EC%99%80-%EC%96%B4%EC%9A%B8%EB%A6%AC%EB%8A%94-%EC%B5%9C%EA%B3%A0%EC%9D%98-%EC%99%80%EC%9D%B8-%EB%AC%B4%EC%97%87%EC%9D%84-%EA%B3%A0%EB%A5%BC-%EA%B2%83%EC%9D%B8', 'content': '### 와인과 각종 주류, 관련 기사를 검색하세요.\n\n마시자 매거진\n마시자 매거진\n\n# 스테이크와 어울리는 최고의 와인: 무엇을 고를 것인가?\n\n# 스테이크와 어울리는 최고의 와인: 무엇을 고를 것인가?\n\n카베르네 소비뇽(Cabernet Sauvignon) 및 말벡(Malbec)과 같은 전형적인 선택부터 더 가벼운 레드 와인, 심지어 화이트 와인과 맛있는 스테이크를 페어링하는 방법까지, 우리의 아카이브에서 가져온 최고의 조언과 최근 디캔터 전문가가 추천한 와인을 소개한다.\n\n<스테이크를 곁들인 레드 와인을 위한 5가지 전형적인 선택>\n\n• 카베르네 소비뇽(Cabernet Sauvignon)  \n• 말벡(Malbec)  \n• 그르나슈/쉬라즈 블렌드(Grenache / Shiraz blends)  \n• 시라/쉬라즈(Syrah / Shiraz)  \n• 산지오베제(Sangiovese)\n\n육즙이 풍부한 스테이크와 맛있는 와인이 있는 저녁 식사는 적어도 고기 애호가들에게 인생의 큰 즐거움일 것이다.\n\n와인과 음식 페어링에서 새로운 시도를 하는 것은 항상 재미있지만, 특별한 스테이크 저녁 식사를 준비할 때 고려해야 할 몇 가지 스타일과 주의사항이 있다.\n\n<스테이크에 곁들이는 레드 와인>\n\n이 포도 품종을 세계 와인 무대에 재등장시키고 고품질 쇠고기에 대한 국가의 명성을 가진 아르헨티나 덕분에, 말벡 레드 와인은 스

In [4]:
# 도구 속성
print("자료형: ")
print(type(web_search))
print("-"*100)

print("name: ")
print(web_search.name)
print("-"*100)

print("description: ")
pprint(web_search.description)
print("-"*100)

print("schema: ")
pprint(web_search.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
----------------------------------------------------------------------------------------------------
name: 
tavily_search_results_json
----------------------------------------------------------------------------------------------------
description: 
('A search engine optimized for comprehensive, accurate, and trusted results. '
 'Useful for when you need to answer questions about current events. Input '
 'should be a search query.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Input for the Tavily tool.',
 'properties': {'query': {'description': 'search query to look up',
                          'title': 'Query',
                          'type': 'string'}},
 'required': ['query'],
 'title': 'TavilyInput',
 'type': 'object'}
----------------------------------------------------------------------------------------------------

`(2) 도구(tool) 호출하기`

In [5]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 웹 검색 도구를 직접 LLM에 바인딩 가능
llm_with_tools = llm.bind_tools(tools=[web_search])

In [6]:
# 도구 호출이 필요 없는 LLM 호출을 수행
query = "안녕하세요."
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 82, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_07be26b212', 'id': 'chatcmpl-E1tX40n0ZCrPDlljxkMf7H2PRj3eX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f65de-5061-7941-aea7-bc87af06985f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 11, 'total_tokens': 93, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
-------------------------------------------------------------------------------------

In [7]:
# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해주세요."
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 91, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_07be26b212', 'id': 'chatcmpl-E1tX5gMIj1uUHF0NN6220R0ZT3CIa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f65de-6a2d-7232-87b1-b90a712120b6-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '스테이크 와인 pairing 추천'}, 'id': 'call_KKJmwlCJegrwiinTJVcJ2Yx9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 91, 'output_tokens': 24, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_tok

In [8]:
tool_call = ai_msg.tool_calls[0]
tool_call

{'name': 'tavily_search_results_json',
 'args': {'query': '스테이크 와인 pairing 추천'},
 'id': 'call_KKJmwlCJegrwiinTJVcJ2Yx9',
 'type': 'tool_call'}

`(3) 도구(tool) 실행하기`

In [9]:
### 방법 1: 직접 도구 호출 처리

# 이 방법은 AI 메시지에서 첫 번째 도구 호출을 가져와 직접 처리한다.
# 'args'를 사용하여 도구를 호출하고 결과를 얻는다.

tool_output = web_search.invoke(tool_call["args"])
print(f"{tool_call['name']} 호출 결과:")
print("-" * 100)
print(tool_output)

tavily_search_results_json 호출 결과:
----------------------------------------------------------------------------------------------------
[{'title': '스테이크에 잘 어울리는 와인 추천 - gogigajoa 님의 블로그', 'url': 'https://gogigajoa.tistory.com/25', 'content': "## 📌 5. 와인 페어링 정리표\n\n스테이크 종류 추천 와인 대표 브랜드\n\n|  |  |  |\n --- \n| 등심, 채끝 (기름진 부위) | 카베르네 소비뇽, 시라 | 샤토 마고, 펜폴즈 그랜지 |\n| 안심 (부드러운 부위) | 피노 누아, 말벡 | 로마네 꽁띠, 까테나 자파타 |\n| 훈제 & 양념 스테이크 | 진판델, 쉬라즈 | 리지 리튼 스프링스, 시걸즈 |\n| 돼지고기 스테이크 | 피노 누아, 샤도네이 | 루이 라투르, 조셉 드루앙 |\n| 닭고기 스테이크 | 샤도네이, 소비뇽 블랑 | 클라우디 베이, 루이 자도 |\n\n## 🔥 최종 정리 – 스테이크 & 와인 완벽 페어링\n\n✅ 기름진 스테이크 (등심, 채끝) → 강한 탄닌 와인 (카베르네 소비뇽, 시라)  \n✅ 부드러운 스테이크 (안심) → 과일 향이 풍부한 와인 (피노 누아, 말벡)  \n✅ 훈제 or 양념 스테이크 → 스파이시한 와인 (템프라니요, 진판델)  \n✅ 돼지고기 & 닭고기 스테이크 → 화이트 와인 or 가벼운 레드 와인\n\n이제 스테이크 & 와인 페어링을 완벽하게 즐길 수 있습니다! 🍷🥩  \n💬 더 궁금한 점 있으면 언제든지 질문 주세요! 😊\n\n#### '소고기에대해서아라보자' 카테고리의 다른 글 [...] 본문 바로가기\n\n# gogigajoa 님의 블로그\n\nPOWERED BY TISTORY\n\n소고기에대해서아라보자\n\n# 🥩 스테이크에 잘 어울리는 와인 추천 🍷\n\ngogigajoa 2025. 3. 14. 13:33\n

In [10]:
### 방법 2: ToolMessage 객체 생성

# 이 방법은 도구 호출 결과를 사용하여 ToolMessage 객체를 생성한다.
# 도구 호출의 ID와 이름을 포함하여 더 구조화된 메시지를 만든다.

from langchain_core.messages import ToolMessage
tool_message = ToolMessage(
    content=tool_output,
    tool_call_id=tool_call["id"],
    name=tool_call["name"]
)

print(tool_message)

content=[{'title': '스테이크에 잘 어울리는 와인 추천 - gogigajoa 님의 블로그', 'url': 'https://gogigajoa.tistory.com/25', 'content': "## 📌 5. 와인 페어링 정리표\n\n스테이크 종류 추천 와인 대표 브랜드\n\n|  |  |  |\n --- \n| 등심, 채끝 (기름진 부위) | 카베르네 소비뇽, 시라 | 샤토 마고, 펜폴즈 그랜지 |\n| 안심 (부드러운 부위) | 피노 누아, 말벡 | 로마네 꽁띠, 까테나 자파타 |\n| 훈제 & 양념 스테이크 | 진판델, 쉬라즈 | 리지 리튼 스프링스, 시걸즈 |\n| 돼지고기 스테이크 | 피노 누아, 샤도네이 | 루이 라투르, 조셉 드루앙 |\n| 닭고기 스테이크 | 샤도네이, 소비뇽 블랑 | 클라우디 베이, 루이 자도 |\n\n## 🔥 최종 정리 – 스테이크 & 와인 완벽 페어링\n\n✅ 기름진 스테이크 (등심, 채끝) → 강한 탄닌 와인 (카베르네 소비뇽, 시라)  \n✅ 부드러운 스테이크 (안심) → 과일 향이 풍부한 와인 (피노 누아, 말벡)  \n✅ 훈제 or 양념 스테이크 → 스파이시한 와인 (템프라니요, 진판델)  \n✅ 돼지고기 & 닭고기 스테이크 → 화이트 와인 or 가벼운 레드 와인\n\n이제 스테이크 & 와인 페어링을 완벽하게 즐길 수 있습니다! 🍷🥩  \n💬 더 궁금한 점 있으면 언제든지 질문 주세요! 😊\n\n#### '소고기에대해서아라보자' 카테고리의 다른 글 [...] 본문 바로가기\n\n# gogigajoa 님의 블로그\n\nPOWERED BY TISTORY\n\n소고기에대해서아라보자\n\n# 🥩 스테이크에 잘 어울리는 와인 추천 🍷\n\ngogigajoa 2025. 3. 14. 13:33\n\n### 🥩 스테이크에 잘 어울리는 와인 추천 🍷\n\n✅ 스테이크의 부위 & 굽기 정도에 맞춘 와인 페어링  \n✅ 레드 와인 추천 (카베르네 소비뇽, 시라, 말벡 등)  \n✅ 화이트 와인 & 기타 와인 추천 (닭고기 & 

In [11]:
### 방법 3: 도구 직접 호출하여 바로 ToolMessage 객체 생성

# 이 방법은 도구를 직접 호출하여 ToolMessage 객체를 생성한다.
# 가장 간단하고 직관적인 방법으로, LangChain의 추상화를 활용한다.

tool_message = web_search.invoke(tool_call)

print(tool_message)


content='[{"title": "스테이크에 잘 어울리는 와인 추천 - gogigajoa 님의 블로그", "url": "https://gogigajoa.tistory.com/25", "content": "## 📌 5. 와인 페어링 정리표\\n\\n스테이크 종류 추천 와인 대표 브랜드\\n\\n|  |  |  |\\n --- \\n| 등심, 채끝 (기름진 부위) | 카베르네 소비뇽, 시라 | 샤토 마고, 펜폴즈 그랜지 |\\n| 안심 (부드러운 부위) | 피노 누아, 말벡 | 로마네 꽁띠, 까테나 자파타 |\\n| 훈제 & 양념 스테이크 | 진판델, 쉬라즈 | 리지 리튼 스프링스, 시걸즈 |\\n| 돼지고기 스테이크 | 피노 누아, 샤도네이 | 루이 라투르, 조셉 드루앙 |\\n| 닭고기 스테이크 | 샤도네이, 소비뇽 블랑 | 클라우디 베이, 루이 자도 |\\n\\n## 🔥 최종 정리 – 스테이크 & 와인 완벽 페어링\\n\\n✅ 기름진 스테이크 (등심, 채끝) → 강한 탄닌 와인 (카베르네 소비뇽, 시라)  \\n✅ 부드러운 스테이크 (안심) → 과일 향이 풍부한 와인 (피노 누아, 말벡)  \\n✅ 훈제 or 양념 스테이크 → 스파이시한 와인 (템프라니요, 진판델)  \\n✅ 돼지고기 & 닭고기 스테이크 → 화이트 와인 or 가벼운 레드 와인\\n\\n이제 스테이크 & 와인 페어링을 완벽하게 즐길 수 있습니다! 🍷🥩  \\n💬 더 궁금한 점 있으면 언제든지 질문 주세요! 😊\\n\\n#### \'소고기에대해서아라보자\' 카테고리의 다른 글 [...] 본문 바로가기\\n\\n# gogigajoa 님의 블로그\\n\\nPOWERED BY TISTORY\\n\\n소고기에대해서아라보자\\n\\n# 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\ngogigajoa 2025. 3. 14. 13:33\\n\\n### 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\n✅ 스테이크의 부위 & 굽기 정도에 맞춘 와인 페어링  \\n✅ 레드 와인 추천 (카베르네 소비뇽, 

In [12]:
pprint(tool_message.tool_call_id)

'call_KKJmwlCJegrwiinTJVcJ2Yx9'


In [13]:
pprint(tool_message.name)

'tavily_search_results_json'


In [14]:
pprint(tool_message.content)

('[{"title": "스테이크에 잘 어울리는 와인 추천 - gogigajoa 님의 블로그", "url": '
 '"https://gogigajoa.tistory.com/25", "content": "## 📌 5. 와인 페어링 정리표\\n\\n스테이크 '
 '종류 추천 와인 대표 브랜드\\n\\n|  |  |  |\\n --- \\n| 등심, 채끝 (기름진 부위) | 카베르네 소비뇽, 시라 | '
 '샤토 마고, 펜폴즈 그랜지 |\\n| 안심 (부드러운 부위) | 피노 누아, 말벡 | 로마네 꽁띠, 까테나 자파타 |\\n| 훈제 & '
 '양념 스테이크 | 진판델, 쉬라즈 | 리지 리튼 스프링스, 시걸즈 |\\n| 돼지고기 스테이크 | 피노 누아, 샤도네이 | 루이 라투르, '
 '조셉 드루앙 |\\n| 닭고기 스테이크 | 샤도네이, 소비뇽 블랑 | 클라우디 베이, 루이 자도 |\\n\\n## 🔥 최종 정리 – '
 '스테이크 & 와인 완벽 페어링\\n\\n✅ 기름진 스테이크 (등심, 채끝) → 강한 탄닌 와인 (카베르네 소비뇽, 시라)  \\n✅ '
 '부드러운 스테이크 (안심) → 과일 향이 풍부한 와인 (피노 누아, 말벡)  \\n✅ 훈제 or 양념 스테이크 → 스파이시한 와인 '
 '(템프라니요, 진판델)  \\n✅ 돼지고기 & 닭고기 스테이크 → 화이트 와인 or 가벼운 레드 와인\\n\\n이제 스테이크 & 와인 '
 '페어링을 완벽하게 즐길 수 있습니다! 🍷🥩  \\n💬 더 궁금한 점 있으면 언제든지 질문 주세요! 😊\\n\\n#### '
 "'소고기에대해서아라보자' 카테고리의 다른 글 [...] 본문 바로가기\\n\\n# gogigajoa 님의 블로그\\n\\nPOWERED "
 'BY TISTORY\\n\\n소고기에대해서아라보자\\n\\n# 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\ngogigajoa '
 '2025. 3. 14. 13:33\\n\\n### 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\n✅ 스테이크의 부위 & 굽기 정

In [15]:
ai_msg.tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': '스테이크 와인 pairing 추천'},
  'id': 'call_KKJmwlCJegrwiinTJVcJ2Yx9',
  'type': 'tool_call'}]

In [16]:
# batch 실행 - 도구 호출이 여러 개인 경우

# tool_messages = web_search.batch([tool_call])

tool_messages = web_search.batch(ai_msg.tool_calls)

print(tool_messages)
print("-" * 100)
pprint(tool_messages[0].content)

[ToolMessage(content='[{"title": "스테이크에 잘 어울리는 와인 추천 - gogigajoa 님의 블로그", "url": "https://gogigajoa.tistory.com/25", "content": "## 📌 5. 와인 페어링 정리표\\n\\n스테이크 종류 추천 와인 대표 브랜드\\n\\n|  |  |  |\\n --- \\n| 등심, 채끝 (기름진 부위) | 카베르네 소비뇽, 시라 | 샤토 마고, 펜폴즈 그랜지 |\\n| 안심 (부드러운 부위) | 피노 누아, 말벡 | 로마네 꽁띠, 까테나 자파타 |\\n| 훈제 & 양념 스테이크 | 진판델, 쉬라즈 | 리지 리튼 스프링스, 시걸즈 |\\n| 돼지고기 스테이크 | 피노 누아, 샤도네이 | 루이 라투르, 조셉 드루앙 |\\n| 닭고기 스테이크 | 샤도네이, 소비뇽 블랑 | 클라우디 베이, 루이 자도 |\\n\\n## 🔥 최종 정리 – 스테이크 & 와인 완벽 페어링\\n\\n✅ 기름진 스테이크 (등심, 채끝) → 강한 탄닌 와인 (카베르네 소비뇽, 시라)  \\n✅ 부드러운 스테이크 (안심) → 과일 향이 풍부한 와인 (피노 누아, 말벡)  \\n✅ 훈제 or 양념 스테이크 → 스파이시한 와인 (템프라니요, 진판델)  \\n✅ 돼지고기 & 닭고기 스테이크 → 화이트 와인 or 가벼운 레드 와인\\n\\n이제 스테이크 & 와인 페어링을 완벽하게 즐길 수 있습니다! 🍷🥩  \\n💬 더 궁금한 점 있으면 언제든지 질문 주세요! 😊\\n\\n#### \'소고기에대해서아라보자\' 카테고리의 다른 글 [...] 본문 바로가기\\n\\n# gogigajoa 님의 블로그\\n\\nPOWERED BY TISTORY\\n\\n소고기에대해서아라보자\\n\\n# 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\ngogigajoa 2025. 3. 14. 13:33\\n\\n### 🥩 스테이크에 잘 어울리는 와인 추천 🍷\\n\\n✅ 스테이크의 부위 & 굽기 정도에 맞춘 와인 페어링  \\n✅ 레드 와인 추

`(4) ToolMessage를 LLM에 전달하여 답변을 생성하기`

In [17]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# 오늘 날짜 설정
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿
prompt = ChatPromptTemplate([
    ("system", f"You are a helpful AI assistant. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# ChatOpenAI 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# LLM에 도구를 바인딩
llm_with_tools = llm.bind_tools(tools=[web_search])

# LLM 체인 생성
llm_chain = prompt | llm_with_tools

# 도구 실행 체인 정의
@chain
def web_search_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    ai_msg = llm_chain.invoke(input_, config=config)
    print("ai_msg: \n", ai_msg)
    print("-"*100)
    tool_msgs = web_search.batch(ai_msg.tool_calls, config=config)
    print("tool_msgs: \n", tool_msgs)
    print("-"*100)
    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

# 체인 실행
response = web_search_chain.invoke("오늘 모엣샹동 샴페인의 가격은 얼마인가요?")

# 응답 출력
pprint(response.content)

ai_msg: 
 content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 114, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fed8e02fdb', 'id': 'chatcmpl-E1tXFfwux9t8MDXtUQGMqAZ7Kn9YP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f65de-93bc-7811-8ecc-767b0936d5ca-0' tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '모엣샹동 샴페인 가격 2026년 7월'}, 'id': 'call_KoDjXmi4cbZSCeKjB8fHLklP', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 114, 'output_tokens': 35, 'total_tokens': 149, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token

### 2-2. 사용자 정의 도구
- @tool decorator를 통해 사용자 정의 도구를 정의할 수 있음

`(1) 도구(tool) 정의하기`

In [18]:
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
from typing import List

# Tool 정의
@tool
def search_web(query: str) -> str:
    """Searches the internet for information that does not exist in the database or for the latest information."""

    tavily_search = TavilySearchResults(max_results=2)
    docs = tavily_search.invoke(query)

    formatted_docs = "\n---\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in docs
        ])

    if len(formatted_docs) > 0:
        return formatted_docs

    return "관련 정보를 찾을 수 없습니다."

In [19]:
# 도구 속성
print("자료형: ")
print(type(search_web))
print("-"*100)

print("name: ")
print(search_web.name)
print("-"*100)

print("description: ")
pprint(search_web.description)
print("-"*100)

print("schema: ")
pprint(search_web.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_core.tools.structured.StructuredTool'>
----------------------------------------------------------------------------------------------------
name: 
search_web
----------------------------------------------------------------------------------------------------
description: 
('Searches the internet for information that does not exist in the database or '
 'for the latest information.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Searches the internet for information that does not exist in '
                'the database or for the latest information.',
 'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'search_web',
 'type': 'object'}
----------------------------------------------------------------------------------------------------


In [20]:
query = "스테이크와 어울리는 와인을 추천해주세요."
search_result = search_web.invoke(query)

print(search_result)

<Document href="https://mashija.com/%EC%8A%A4%ED%85%8C%EC%9D%B4%ED%81%AC%EC%99%80-%EC%96%B4%EC%9A%B8%EB%A6%AC%EB%8A%94-%EC%B5%9C%EA%B3%A0%EC%9D%98-%EC%99%80%EC%9D%B8-%EB%AC%B4%EC%97%87%EC%9D%84-%EA%B3%A0%EB%A5%BC-%EA%B2%83%EC%9D%B8"/>
### 와인과 각종 주류, 관련 기사를 검색하세요.

마시자 매거진
마시자 매거진

# 스테이크와 어울리는 최고의 와인: 무엇을 고를 것인가?

# 스테이크와 어울리는 최고의 와인: 무엇을 고를 것인가?

카베르네 소비뇽(Cabernet Sauvignon) 및 말벡(Malbec)과 같은 전형적인 선택부터 더 가벼운 레드 와인, 심지어 화이트 와인과 맛있는 스테이크를 페어링하는 방법까지, 우리의 아카이브에서 가져온 최고의 조언과 최근 디캔터 전문가가 추천한 와인을 소개한다.

<스테이크를 곁들인 레드 와인을 위한 5가지 전형적인 선택>

• 카베르네 소비뇽(Cabernet Sauvignon)  
• 말벡(Malbec)  
• 그르나슈/쉬라즈 블렌드(Grenache / Shiraz blends)  
• 시라/쉬라즈(Syrah / Shiraz)  
• 산지오베제(Sangiovese)

육즙이 풍부한 스테이크와 맛있는 와인이 있는 저녁 식사는 적어도 고기 애호가들에게 인생의 큰 즐거움일 것이다.

와인과 음식 페어링에서 새로운 시도를 하는 것은 항상 재미있지만, 특별한 스테이크 저녁 식사를 준비할 때 고려해야 할 몇 가지 스타일과 주의사항이 있다.

<스테이크에 곁들이는 레드 와인>

이 포도 품종을 세계 와인 무대에 재등장시키고 고품질 쇠고기에 대한 국가의 명성을 가진 아르헨티나 덕분에, 말벡 레드 와인은 스테이크와 함께 고전적인 매칭이 되었다.

말벡의 풍부한 짙은 과일의 특징과 자연스러운 타닌은 일반적으로 좋은 스테이크와 잘 어울린다고 여겨지지만, 

In [21]:
# LLM에 도구를 바인딩
llm_with_tools = llm.bind_tools(tools=[search_web])

# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해주세요."
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 67, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_469628dc88', 'id': 'chatcmpl-E1tXSMKK5MYbMrUFwlrSn9emajXpa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f65de-c4b7-7a22-9b92-eb285bfe290b-0', tool_calls=[{'name': 'search_web', 'args': {'query': '스테이크와 어울리는 와인 추천'}, 'id': 'call_HKHUnEyFaEIFMZvuHpv3GFjw', 'type': 'tool_call'}, {'name': 'search_web', 'args': {'query': 'red wine pairings with steak'}, 'id': 'call_cwRV55NGgxYzzeCEctWIJuDN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tok

`(2) LLM 도구 호출 성능 비교하기`

In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 기본 LLM
llm_gemini_flash = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
llm_gemini_pro = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)
llm_groq = ChatGroq(model="llama3-70b-8192", temperature=0)

# LLM에 도구 바인딩하여 추가
tools=[search_web]

gemini_flash_with_tools = llm_gemini_flash.bind_tools(tools)
gemini_pro_with_tools = llm_gemini_pro.bind_tools(tools)
groq_llama3_with_tools = llm_groq.bind_tools(tools)

ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-1.5-fla...: 0, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

- gemini-1.5-flash

In [51]:
# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해주세요."
ai_msg = gemini_flash_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

NameError: name 'gemini_flash_with_tools' is not defined

- gemini-1.5-pro

In [52]:
# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해주세요."
ai_msg = gemini_pro_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

NameError: name 'gemini_pro_with_tools' is not defined

- llama3-70b-8192

In [53]:
# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해주세요."
ai_msg = groq_llama3_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

NameError: name 'groq_llama3_with_tools' is not defined

### 2-3. Runnable 객체를 도구(tool) 변환
- 문자열이나 dict 입력을 받는 Runnable을 도구로 변환
- as_tool 메서드를 사용

`(1) Document Loader`

In [22]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field
from typing import List

# WikipediaLoader를 사용하여 위키피디아 문서를 검색하는 함수
def search_wiki(input_data: dict) -> List[Document]:
    """Search Wikipedia documents based on user input (query) and return k documents"""
    query = input_data["query"]
    k = input_data.get("k", 2)
    wiki_loader = WikipediaLoader(query=query, load_max_docs=k, lang="ko")
    wiki_docs = wiki_loader.load()
    return wiki_docs

# 도구 호출에 사용할 입력 스키마 정의
class WikiSearchSchema(BaseModel):
    """Input schema for Wikipedia search."""
    query: str = Field(..., description="The query to search for in Wikipedia")
    k: int = Field(2, description="The number of documents to return (default is 2)")

# RunnableLambda 함수를 사용하여 위키피디아 문서 로더를 Runnable로 변환
runnable = RunnableLambda(search_wiki)
wiki_search = runnable.as_tool(
    name="wiki_search",
    description=dedent("""
        Use this tool when you need to search for information on Wikipedia.
        It searches for Wikipedia articles related to the user's query and returns
        a specified number of documents. This tool is useful when general knowledge
        or background information is required.
    """),
    args_schema=WikiSearchSchema
)

/tmp/ipykernel_1261816/1492815655.py:24: LangChainBetaWarning: This API is in beta and may change in the future.
  wiki_search = runnable.as_tool(


In [23]:
# 도구 속성
print("자료형: ")
print(type(wiki_search))
print("-"*100)

print("name: ")
print(wiki_search.name)
print("-"*100)

print("description: ")
pprint(wiki_search.description)
print("-"*100)

print("schema: ")
pprint(wiki_search.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_core.tools.structured.StructuredTool'>
----------------------------------------------------------------------------------------------------
name: 
wiki_search
----------------------------------------------------------------------------------------------------
description: 
('Use this tool when you need to search for information on Wikipedia.\n'
 "It searches for Wikipedia articles related to the user's query and returns\n"
 'a specified number of documents. This tool is useful when general knowledge\n'
 'or background information is required.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Input schema for Wikipedia search.',
 'properties': {'k': {'default': 2,
                      'description': 'The number of documents to return '
                                     '(default is 2)',
                      'title': 'K',
                      'type': 'integer'},
                'q

In [24]:
# 위키 검색 실행
query = "파스타의 유래"
wiki_results = wiki_search.invoke({"query":query})

# 검색 결과 출력
for result in wiki_results:
    print(result)
    print("-" * 100)

page_content='오르조(이탈리아어: orzo) 또는 리소니(이탈리아어: risoni, 단수: risone 리소네[*])는 이탈리아의 파스타이다. "오르조"는 라틴어:hordeum에서 유래했으며 "보리"를 뜻한다. "리소니"는 "큰 쌀"이라는 뜻이다. 파스타의 일종으로 큰 쌀알의 모양을 하고 있으며 솔방울이나 잣보다는 좀 더 작다. 보통 라구 등 수프와 함께 먹는다. 원래는 보리로 만들었지만 요즘에는 박력분으로 만드는 것이 흔해졌다. 다른 이름으로는 kritharáki ("little barley") 혹은 manéstra(그리스 요리)로 부르며, lisān al-`uṣfūr ("명금의 혀")라고 아랍 요리에서 불린다. 오르조를 두고 이탈리아의 쌀이라고 부르기도 한다.
터키에서는 아르파 셰흐리예(arpa şehriye)로 불리며, 셰흐리예의 하나이다. 필라브를 만들거나 초르바(수프)에 넣어 먹는다.


== 같이 보기 ==
파스티나
셰흐리예


== 각주 ==' metadata={'title': '오르조', 'summary': '오르조(이탈리아어: orzo) 또는 리소니(이탈리아어: risoni, 단수: risone 리소네[*])는 이탈리아의 파스타이다. "오르조"는 라틴어:hordeum에서 유래했으며 "보리"를 뜻한다. "리소니"는 "큰 쌀"이라는 뜻이다. 파스타의 일종으로 큰 쌀알의 모양을 하고 있으며 솔방울이나 잣보다는 좀 더 작다. 보통 라구 등 수프와 함께 먹는다. 원래는 보리로 만들었지만 요즘에는 박력분으로 만드는 것이 흔해졌다. 다른 이름으로는 kritharáki ("little barley") 혹은 manéstra(그리스 요리)로 부르며, lisān al-`uṣfūr ("명금의 혀")라고 아랍 요리에서 불린다. 오르조를 두고 이탈리아의 쌀이라고 부르기도 한다.\n터키에서는 아르파 셰흐리예(arpa şehriye)로 불리며, 셰흐리예의 하나이다. 필라브를 만들거나 초르바(수프)에 넣어 먹는다.', 'source': 'https://ko.wik

In [25]:
# LLM에 도구를 바인딩 (2개의 도구 바인딩)
llm_with_tools = llm.bind_tools(tools=[search_web, wiki_search])

# 도구 호출이 필요한 LLM 호출을 수행
query = "서울 강남의 유명한 파스타 맛집은 어디인가요? 그리고 파스타의 유래를 알려주세요. "
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 174, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8516364442', 'id': 'chatcmpl-E1tXY8VcpmxFI7B0E78DFKg4vOQIB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f65de-dd27-7780-b507-85f74226b03c-0', tool_calls=[{'name': 'search_web', 'args': {'query': '서울 강남 유명 파스타 맛집'}, 'id': 'call_QKDi9eK2cyMv4MxpwPgD9gRY', 'type': 'tool_call'}, {'name': 'wiki_search', 'args': {'query': '파스타 유래'}, 'id': 'call_7aiB0fJFq4HD2EdXXZcIz5yr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 174, 'output_to

`(2) LCEL 체인`
- 위키피디아 문서를 검색하고 내용을 요약하는 체인

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia.wikipedia as wikipedia_client

# Wikipedia API를 사용하여 위키피디아 문서를 검색하고 텍스트로 반환하는 함수
def wiki_search_and_summarize(input_data: dict):
    wiki = WikipediaAPIWrapper(top_k_results=2, lang="ko")
    wikipedia_client.API_URL = "https://ko.wikipedia.org/w/api.php"
    wikipedia_client.set_user_agent("docreview-rag-agent/1.0 (educational LangChain notebook)")
    wiki_docs = wiki.load(input_data["query"])

    formatted_docs =[
        f'<Document source="{doc.metadata["source"]}"/>\n{doc.page_content}\n</Document>'
        for doc in wiki_docs
        ]

    return formatted_docs

# 요약 프롬프트 템플릿
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text in a concise manner:\n\n{context}\n\nSummary:"
)

# LLM 및 요약 체인 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
summary_chain = (
    {"context": RunnableLambda(wiki_search_and_summarize)}
    | summary_prompt | llm | StrOutputParser()
)

# 요약 테스트
summarized_text = summary_chain.invoke({"query":"파스타의 유래"})
pprint(summarized_text)

('오르조(리소니)는 이탈리아의 파스타로, 보리에서 유래한 이름을 가지고 있으며 큰 쌀알 모양이다. 주로 수프와 함께 먹으며, 현재는 '
 '박력분으로 만들어진다. 터키에서는 아르파 셰흐리예로 불린다. \n'
 '\n'
 '파파르델레는 넓고 큰 형태의 이탈리아 파스타로, "게걸스럽게 먹다"는 뜻의 이탈리아어에서 유래했다. 신선한 파파르델레는 약 3cm '
 '너비이며, 주로 돼지고기와 토끼고기 육수로 조리된다. 이 파스타는 이탈리아의 축제에서도 중요한 역할을 한다.')


In [33]:
# 도구 호출에 사용할 입력 스키마 정의
class WikiSummarySchema(BaseModel):
    """Input schema for Wikipedia search."""
    query: str = Field(..., description="The query to search for in Wikipedia")

# as_tool 메소드를 사용하여 도구 객체로 변환
wiki_summary = summary_chain.as_tool(
    name="wiki_summary",
    description=dedent("""
        Use this tool when you need to search for information on Wikipedia.
        It searches for Wikipedia articles related to the user's query and returns
        a summarized text. This tool is useful when general knowledge
        or background information is required.
    """),
    args_schema=WikiSummarySchema
)

# 도구 속성
print("자료형: ")
print(type(wiki_summary))
print("-"*100)

print("name: ")
print(wiki_summary.name)
print("-"*100)

print("description: ")
pprint(wiki_summary.description)
print("-"*100)

print("schema: ")
pprint(wiki_summary.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_core.tools.structured.StructuredTool'>
----------------------------------------------------------------------------------------------------
name: 
wiki_summary
----------------------------------------------------------------------------------------------------
description: 
('Use this tool when you need to search for information on Wikipedia.\n'
 "It searches for Wikipedia articles related to the user's query and returns\n"
 'a summarized text. This tool is useful when general knowledge\n'
 'or background information is required.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Input schema for Wikipedia search.',
 'properties': {'query': {'description': 'The query to search for in Wikipedia',
                          'title': 'Query',
                          'type': 'string'}},
 'required': ['query'],
 'title': 'WikiSummarySchema',
 'type': 'object'}
-----------------------------

In [34]:
# LLM에 도구를 바인딩
llm_with_tools = llm.bind_tools(tools=[search_web, wiki_summary])

# 도구 호출이 필요한 LLM 호출을 수행
query = "서울 강남의 유명한 파스타 맛집은 어디인가요? 그리고 파스타의 유래를 알려주세요. "
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 150, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_854585a3aa', 'id': 'chatcmpl-E1tcL2SXbVhdNrV4cHFoAJYiXxXmr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f65e3-66d8-7972-a509-495399434ae9-0', tool_calls=[{'name': 'search_web', 'args': {'query': '서울 강남 파스타 맛집 추천'}, 'id': 'call_Y55zR9u6woXfjRO5CdeljVym', 'type': 'tool_call'}, {'name': 'wiki_summary', 'args': {'query': '파스타'}, 'id': 'call_tLFksPqD5HkNGcwXZRbTzWir', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_toke

In [35]:
ai_msg.tool_calls[1]

{'name': 'wiki_summary',
 'args': {'query': '파스타'},
 'id': 'call_tLFksPqD5HkNGcwXZRbTzWir',
 'type': 'tool_call'}

In [36]:
# 도구 실행
tool_message = wiki_summary.invoke(ai_msg.tool_calls[1])

print(tool_message)
print("-" * 100)
pprint(tool_message.content)

content='The text provides two distinct summaries: \n\n1. **Pasta**: Pasta, an Italian staple made from durum wheat semolina mixed with water or eggs, is a non-leavened food typically cooked by boiling or baking. The term "pasta" means "dough" in Italian. Its history dates back to ancient times, with references to similar foods in Greek and Arabic texts. There are two main types: dried pasta, which is made from durum wheat and can be stored for long periods, and fresh pasta, made from soft wheat and eggs, often prepared for special occasions. Pasta comes in various shapes and is typically served with different sauces.\n\n2. **Lee Sun-kyun**: Lee Sun-kyun (1975-2023) was a South Korean actor known for his roles in films like "Parasite," for which he received critical acclaim and awards. He began his career in musicals and gained popularity through various television dramas. Lee faced legal issues related to drug use before his tragic death by suicide in December 2023. He was married to 

In [38]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# 오늘 날짜 설정
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿
prompt = ChatPromptTemplate([
    ("system", f"You are a helpful AI assistant. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# LLM에 도구를 바인딩
llm_with_tools = llm.bind_tools(tools=[wiki_summary])

# LLM 체인 생성
llm_chain = prompt | llm_with_tools

# 도구 실행 체인 정의
@chain
def wiki_summary_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    ai_msg = llm_chain.invoke(input_, config=config)
    print("ai_msg: \n", ai_msg)
    print("-"*100)
    tool_msgs = wiki_summary.batch(ai_msg.tool_calls, config=config)
    print("tool_msgs: \n", tool_msgs)
    print("-"*100)
    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

# 체인 실행
response = wiki_summary_chain.invoke("파스타의 유래에 대해서 알려주세요.")

# 응답 출력
pprint(response.content)

ai_msg: 
 content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 120, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_13015c0152', 'id': 'chatcmpl-E1tdPZ374tmVlWOZzozFlKz79JnfF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f65e4-664c-7520-8625-b05c508906d5-0' tool_calls=[{'name': 'wiki_summary', 'args': {'query': '파스타의 유래'}, 'id': 'call_HVX4GpDEZ1J9OR3oUb40j2OM', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 120, 'output_tokens': 19, 'total_tokens': 139, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

### 2-4. 벡터저장소 검색기
- @tool decorator 사용

`(1) 문서 로드 및 인덱싱`

In [41]:
from langchain_community.document_loaders import TextLoader

# 메뉴판 텍스트 데이터를 로드
loader = TextLoader("./data/restaurant_menu.txt", encoding="utf-8")
documents = loader.load()

print(len(documents))

1


In [42]:
from langchain_core.documents import Document

# 문서 분할 (Chunking)
def split_menu_items(document):
    """
    메뉴 항목을 분리하는 함수
    """
    # 정규표현식 정의
    pattern = r'(\d+\.\s.*?)(?=\n\n\d+\.|$)'
    menu_items = re.findall(pattern, document.page_content, re.DOTALL)

    # 각 메뉴 항목을 Document 객체로 변환
    menu_documents = []
    for i, item in enumerate(menu_items, 1):
        # 메뉴 이름 추출
        menu_name = item.split('\n')[0].split('.', 1)[1].strip()

        # 새로운 Document 객체 생성
        menu_doc = Document(
            page_content=item.strip(),
            metadata={
                "source": document.metadata['source'],
                "menu_number": i,
                "menu_name": menu_name
            }
        )
        menu_documents.append(menu_doc)

    return menu_documents


# 메뉴 항목 분리 실행
menu_documents = []
for doc in documents:
    menu_documents += split_menu_items(doc)

# 결과 출력
print(f"총 {len(menu_documents)}개의 메뉴 항목이 처리되었습니다.")
for doc in menu_documents[:2]:
    print(f"\n메뉴 번호: {doc.metadata['menu_number']}")
    print(f"메뉴 이름: {doc.metadata['menu_name']}")
    print(f"내용:\n{doc.page_content[:100]}...")

총 10개의 메뉴 항목이 처리되었습니다.

메뉴 번호: 1
메뉴 이름: 시그니처 스테이크
내용:
1. 시그니처 스테이크
   • 가격: ₩35,000
   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스
   • 설명: 셰프의 특제 시그니처 메뉴로, ...

메뉴 번호: 2
메뉴 이름: 트러플 리조또
내용:
2. 트러플 리조또
   • 가격: ₩22,000
   • 주요 식재료: 이탈리아산 아르보리오 쌀, 블랙 트러플, 파르미지아노 레지아노 치즈
   • 설명: 크리미한 텍스처의 리조...


In [43]:
# Chroma Vectorstore를 사용하기 위한 준비
from langchain_chroma import Chroma
from langchain_ollama  import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="bge-m3")

# Chroma 인덱스 생성
menu_db = Chroma.from_documents(
    documents=menu_documents,
    embedding=embeddings_model,
    collection_name="restaurant_menu",
    persist_directory="./chroma_db",
)

# Retriever 생성
menu_retriever = menu_db.as_retriever(
    search_kwargs={'k': 2},
)

# 쿼리 테스트
query = "시그니처 스테이크의 가격과 특징은 무엇인가요?"
docs = menu_retriever.invoke(query)
print(f"검색 결과: {len(docs)}개")

for doc in docs:
    print(f"메뉴 번호: {doc.metadata['menu_number']}")
    print(f"메뉴 이름: {doc.metadata['menu_name']}")
    print()

검색 결과: 2개
메뉴 번호: 1
메뉴 이름: 시그니처 스테이크

메뉴 번호: 8
메뉴 이름: 안심 스테이크 샐러드



- 와인 메뉴에 대해서도 같은 작업을 처리

In [44]:
# 와인 메뉴 텍스트 데이터를 로드
loader = TextLoader("./data/restaurant_wine.txt", encoding="utf-8")
documents = loader.load()

# 메뉴 항목 분리 실행
menu_documents = []
for doc in documents:
    menu_documents += split_menu_items(doc)

# 결과 출력
print(f"총 {len(menu_documents)}개의 메뉴 항목이 처리되었습니다.")
for doc in menu_documents[:2]:
    print(f"\n메뉴 번호: {doc.metadata['menu_number']}")
    print(f"메뉴 이름: {doc.metadata['menu_name']}")
    print(f"내용:\n{doc.page_content[:100]}...")


# Chroma 인덱스 생성
wine_db = Chroma.from_documents(
    documents=menu_documents,
    embedding=embeddings_model,
    collection_name="restaurant_wine",
    persist_directory="./chroma_db",
)

wine_retriever = wine_db.as_retriever(
    search_kwargs={'k': 2},
)

query = "스테이크와 어울리는 와인을 추천해주세요."
docs = wine_retriever.invoke(query)
print(f"검색 결과: {len(docs)}개")

for doc in docs:
    print(f"메뉴 번호: {doc.metadata['menu_number']}")
    print(f"메뉴 이름: {doc.metadata['menu_name']}")
    print()

총 10개의 메뉴 항목이 처리되었습니다.

메뉴 번호: 1
메뉴 이름: 샤토 마고 2015
내용:
1. 샤토 마고 2015
   • 가격: ₩450,000
   • 주요 품종: 카베르네 소비뇽, 메를로, 카베르네 프랑, 쁘띠 베르도
   • 설명: 보르도 메독 지역의 프리미엄 ...

메뉴 번호: 2
메뉴 이름: 돔 페리뇽 2012
내용:
2. 돔 페리뇽 2012
   • 가격: ₩380,000
   • 주요 품종: 샤르도네, 피노 누아
   • 설명: 프랑스 샴페인의 대명사로 알려진 프레스티지 큐베입니다. 시트러스...
검색 결과: 2개
메뉴 번호: 6
메뉴 이름: 바롤로 몬프리바토 2017

메뉴 번호: 7
메뉴 이름: 풀리니 몽라쉐 1er Cru 2018



`(2) 도구(tool) 정의하기`

In [46]:
# 벡터 저장소 로드
menu_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="restaurant_menu",
    persist_directory="./chroma_db",
)

@tool
def search_menu(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant menu information from the encrypted database.
    Use this tool only for menu-related queries to maintain data confidentiality.
    """
    docs = menu_db.similarity_search(query, k=2)
    if len(docs) > 0:
        return docs

    return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]

# 도구 속성
print("자료형: ")
print(type(search_menu))
print("-"*100)

print("name: ")
print(search_menu.name)
print("-"*100)

print("description: ")
pprint(search_menu.description)
print("-"*100)

print("schema: ")
pprint(search_menu.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_core.tools.structured.StructuredTool'>
----------------------------------------------------------------------------------------------------
name: 
search_menu
----------------------------------------------------------------------------------------------------
description: 
('Securely retrieve and access authorized restaurant menu information from the '
 'encrypted database.\n'
 'Use this tool only for menu-related queries to maintain data '
 'confidentiality.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Securely retrieve and access authorized restaurant menu '
                'information from the encrypted database.\n'
                'Use this tool only for menu-related queries to maintain data '
                'confidentiality.',
 'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'search_menu',
 'type': 'object'}
----------------

In [47]:
from langchain_core.tools import tool
from typing import List
from langchain_core.documents import Document

# 벡터 저장소 로드
wine_db = Chroma(
   embedding_function=embeddings_model,
   collection_name="restaurant_wine",
   persist_directory="./chroma_db",
)

@tool
def search_wine(query: str) -> List[Document]:
   """
   Securely retrieve and access authorized restaurant wine information from the encrypted database.
   Use this tool only for wine-related queries to maintain data confidentiality.
   """
   docs = wine_db.similarity_search(query, k=2)
   if len(docs) > 0:
      return docs

   return [Document(page_content="관련 와인 정보를 찾을 수 없습니다.")]

# 도구 속성
print("자료형: ")
print(type(search_wine))
print("-"*100)

print("name: ")
print(search_wine.name)
print("-"*100)

print("description: ")
pprint(search_wine.description)
print("-"*100)

print("schema: ")
pprint(search_wine.args_schema.schema())
print("-"*100)

자료형: 
<class 'langchain_core.tools.structured.StructuredTool'>
----------------------------------------------------------------------------------------------------
name: 
search_wine
----------------------------------------------------------------------------------------------------
description: 
('Securely retrieve and access authorized restaurant wine information from the '
 'encrypted database.\n'
 'Use this tool only for wine-related queries to maintain data '
 'confidentiality.')
----------------------------------------------------------------------------------------------------
schema: 
{'description': 'Securely retrieve and access authorized restaurant wine '
                'information from the encrypted database.\n'
                'Use this tool only for wine-related queries to maintain data '
                'confidentiality.',
 'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'search_wine',
 'type': 'object'}
----------------

In [48]:
# LLM에 도구를 바인딩 (2개의 도구 바인딩)
llm_with_tools = llm.bind_tools(tools=[search_menu, search_wine])

# 도구 호출이 필요한 LLM 호출을 수행
query = "시그니처 스테이크의 가격과 특징은 무엇인가요? 그리고 스테이크와 어울리는 와인 추천도 해주세요."
ai_msg = llm_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 137, 'total_tokens': 190, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_07be26b212', 'id': 'chatcmpl-E1tguJCqWVqU9F6XMs43fvrn33Z8H', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f65e7-b54c-78e0-a123-eab20fa5b25c-0', tool_calls=[{'name': 'search_menu', 'args': {'query': '시그니처 스테이크'}, 'id': 'call_5dBk6yPp288Tkt7Nic4BOyus', 'type': 'tool_call'}, {'name': 'search_wine', 'args': {'query': '스테이크'}, 'id': 'call_fav0apZUKbOoBdzs6cHSM7XN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 

`(3) 여러 개의 도구(tool) 호출하기`

In [49]:
tools = [search_web, wiki_summary, search_wine, search_menu]
for tool in tools:
    print(tool.name)

search_web
wiki_summary
search_wine
search_menu


In [50]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# 오늘 날짜 설정
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿
prompt = ChatPromptTemplate([
    ("system", f"You are a helpful AI assistant. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# ChatOpenAI 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 4개의 검색 도구를 LLM에 바인딩
llm_with_tools = llm.bind_tools(tools=tools)

# LLM 체인 생성
llm_chain = prompt | llm_with_tools

# 도구 실행 체인 정의
@chain
def restaurant_menu_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    ai_msg = llm_chain.invoke(input_, config=config)

    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        print(f"{tool_call['name']}: \n{tool_call}")
        print("-"*100)

        if tool_call["name"] == "search_web":
            tool_message = search_web.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "wiki_summary":
            tool_message = wiki_summary.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "search_wine":
            tool_message = search_wine.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "search_menu":
            tool_message = search_menu.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

    print("tool_msgs: \n", tool_msgs)
    print("-"*100)
    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

# 체인 실행
response = restaurant_menu_chain.invoke("시그니처 스테이크의 가격과 특징은 무엇인가요? 그리고 스테이크와 어울리는 와인 추천도 해주세요.")

# 응답 출력
print(response.content)

search_menu: 
{'name': 'search_menu', 'args': {'query': '시그니처 스테이크'}, 'id': 'call_bfMmxno5EkYPLIqBBTE4L6UC', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
search_wine: 
{'name': 'search_wine', 'args': {'query': '스테이크'}, 'id': 'call_WIqECEcxcay2uH0LKk1HZFAV', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
tool_msgs: 
 [ToolMessage(content="[Document(id='55420428-0f94-4f82-9383-4e2e3c4cea62', metadata={'menu_number': 1, 'menu_name': '시그니처 스테이크', 'source': './data/restaurant_menu.txt'}, page_content='1. 시그니처 스테이크\\n   • 가격: ₩35,000\\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.'), Document(id='02b0713d-fff7-48fd-ba37-b820c174fffd', metadata={'menu_name': '안심 스테이크 샐러드', 'source': './data/restau

In [52]:
# 체인 실행
response = restaurant_menu_chain.invoke("파스타 메뉴가 있나요? 이 음식의 역사 또는 유래를 알려주세요.")

# 응답 출력
print(response.content)

search_menu: 
{'name': 'search_menu', 'args': {'query': '파스타'}, 'id': 'call_GgWNuSGrwA1V9zAKPDxZmoux', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
wiki_summary: 
{'name': 'wiki_summary', 'args': {'query': '파스타'}, 'id': 'call_Ss4gqzhxdFHHfKcLVfZlCu02', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
tool_msgs: 
 [ToolMessage(content="[Document(id='0d31bb83-6e20-48b1-bd8c-55935cc2a93a', metadata={'menu_name': '해산물 파스타', 'source': './data/restaurant_menu.txt', 'menu_number': 6}, page_content='6. 해산물 파스타\\n   • 가격: ₩24,000\\n   • 주요 식재료: 링귀네 파스타, 새우, 홍합, 오징어, 토마토 소스\\n   • 설명: 알 덴테로 삶은 링귀네 파스타에 신선한 해산물을 듬뿍 올린 메뉴입니다. 토마토 소스의 산미와 해산물의 감칠맛이 조화를 이루며, 마늘과 올리브 오일로 풍미를 더했습니다. 파슬리를 뿌려 향긋한 맛을 더합니다.'), Document(id='77ef167b-abf4-448c-bbf0-03b0e335f1b3', metadata={'menu_name': '랍스터 비스크', 'source': './data/restaurant_menu.txt', 'menu_number': 7}, page_

## 3. Few-shot 프롬프팅 
- 각 도구의 용도를 구분하여 few-shot 예제로 제시

### 3-1. Few-shot 도구 호출

In [53]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate

examples = [
    HumanMessage("트러플 리조또의 가격과 특징, 그리고 어울리는 와인에 대해 알려주세요.", name="example_user"),
    AIMessage("메뉴 정보를 검색하고, 위키피디아에서 추가 정보를 찾은 후, 어울리는 와인을 검색해보겠습니다.", name="example_assistant"),
    AIMessage("", name="example_assistant", tool_calls=[{"name": "search_menu", "args": {"query": "트러플 리조또"}, "id": "1"}]),
    ToolMessage("트러플 리조또: 가격 ₩28,000, 이탈리아 카나롤리 쌀 사용, 블랙 트러플 향과 파르메산 치즈를 듬뿍 넣어 조리", tool_call_id="1"),
    AIMessage("트러플 리조또의 가격은 ₩28,000이며, 이탈리아 카나롤리 쌀을 사용하고 블랙 트러플 향과 파르메산 치즈를 듬뿍 넣어 조리합니다. 이제 추가 정보를 위키피디아에서 찾아보겠습니다.", name="example_assistant"),
    AIMessage("", name="example_assistant", tool_calls=[{"name": "wiki_summary", "args": {"query": "트러플 리조또", "k": 1}, "id": "2"}]),
    ToolMessage("트러플 리조또는 이탈리아 요리의 대표적인 리조또 요리 중 하나로, 고급 식재료인 트러플을 사용하여 만든 크리미한 쌀 요리입니다. 주로 아르보리오나 카나롤리 등의 쌀을 사용하며, 트러플 오일이나 생 트러플을 넣어 조리합니다. 리조또 특유의 크리미한 질감과 트러플의 강렬하고 독특한 향이 조화를 이루는 것이 특징입니다.", tool_call_id="2"),
    AIMessage("트러플 리조또의 특징에 대해 알아보았습니다. 이제 어울리는 와인을 검색해보겠습니다.", name="example_assistant"),
    AIMessage("", name="example_assistant", tool_calls=[{"name": "search_wine", "args": {"query": "트러플 리조또에 어울리는 와인"}, "id": "3"}]),
    ToolMessage("트러플 리조또와 잘 어울리는 와인으로는 주로 중간 바디의 화이트 와인이 추천됩니다. 1. 샤르도네: 버터와 오크향이 트러플의 풍미를 보완합니다. 2. 피노 그리지오: 산뜻한 산미가 리조또의 크리미함과 균형을 이룹니다. 3. 베르나차: 이탈리아 토스카나 지방의 화이트 와인으로, 미네랄리티가 트러플과 잘 어울립니다.", tool_call_id="3"),
    AIMessage("트러플 리조또(₩28,000)는 이탈리아의 대표적인 리조또 요리 중 하나로, 이탈리아 카나롤리 쌀을 사용하고 블랙 트러플 향과 파르메산 치즈를 듬뿍 넣어 조리합니다. 주요 특징으로는 크리미한 질감과 트러플의 강렬하고 독특한 향이 조화를 이루는 점입니다. 고급 식재료인 트러플을 사용해 풍부한 맛과 향을 내며, 주로 아르보리오나 카나롤리 등의 쌀을 사용합니다. 트러플 리조또와 잘 어울리는 와인으로는 중간 바디의 화이트 와인이 추천됩니다. 특히 버터와 오크향이 트러플의 풍미를 보완하는 샤르도네, 산뜻한 산미로 리조또의 크리미함과 균형을 이루는 피노 그리지오, 그리고 미네랄리티가 트러플과 잘 어울리는 이탈리아 토스카나 지방의 베르나차 등이 좋은 선택이 될 수 있습니다.", name="example_assistant"),
]

system = """You are an AI assistant providing restaurant menu information and general food-related knowledge.
For information about the restaurant's menu, use the search_menu tool.
For other general information, use the wiki_summary tool.
For wine recommendations or pairing information, use the search_wine tool.
If additional web searches are needed or for the most up-to-date information, use the search_web tool.
"""

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    *examples,
    ("human", "{query}"),
])

# ChatOpenAI 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 검색 도구를 직접 LLM에 바인딩 가능
llm_with_tools = llm.bind_tools(tools=tools)

# Few-shot 프롬프트를 사용한 체인 구성
fewshot_search_chain = few_shot_prompt | llm_with_tools

# 체인 실행
query = "스테이크 메뉴가 있나요? 스테이크와 어울리는 와인을 추천해주세요."
response = fewshot_search_chain.invoke(query)

# 결과 출력
for tool_call in response.tool_calls:
    print(tool_call)

{'name': 'search_menu', 'args': {'query': '스테이크'}, 'id': 'call_KB4SDhodnWJMfX9xqAW1h00G', 'type': 'tool_call'}
{'name': 'search_wine', 'args': {'query': '스테이크'}, 'id': 'call_eFyN9LgbpJWW3iUXykkK9zHY', 'type': 'tool_call'}


In [54]:
# 체인 실행
query = "파스타의 유래에 대해서 알고 있나요? 서울 강남의 파스타 맛집을 추천해주세요."
response = fewshot_search_chain.invoke(query)

# 결과 출력
for tool_call in response.tool_calls:
    print(tool_call)

{'name': 'wiki_summary', 'args': {'query': '파스타의 유래'}, 'id': 'call_l8XaWyLcjFxyRX4idagBmTml', 'type': 'tool_call'}
{'name': 'search_web', 'args': {'query': '서울 강남 파스타 맛집 추천'}, 'id': 'call_6fVCoVIOqGmqbSZOop5OHufI', 'type': 'tool_call'}


### 3-2. 답변 생성 체인 

In [55]:
from datetime import datetime
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain
from langchain_openai import ChatOpenAI

# 오늘 날짜 설정
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿
system = """You are an AI assistant providing restaurant menu information and general food-related knowledge.
For information about the restaurant's menu, use the search_menu tool.
For other general information, use the wiki_summary tool.
For wine recommendations or pairing information, use the search_wine tool.
If additional web searches are needed or for the most up-to-date information, use the search_web tool.
"""

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", system + f"Today's date is {today}."),
    *examples,
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# ChatOpenAI 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 검색 도구를 직접 LLM에 바인딩 가능
llm_with_tools = llm.bind_tools(tools=tools)

# Few-shot 프롬프트를 사용한 체인 구성
fewshot_search_chain = few_shot_prompt | llm_with_tools

# 도구 실행 체인 정의
@chain
def restaurant_menu_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    ai_msg = llm_chain.invoke(input_, config=config)

    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        print(f"{tool_call['name']}: \n{tool_call}")
        print("-"*100)

        if tool_call["name"] == "search_web":
            tool_message = search_web.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "wiki_summary":
            tool_message = wiki_summary.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "search_wine":
            tool_message = search_wine.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

        elif tool_call["name"] == "search_menu":
            tool_message = search_menu.invoke(tool_call, config=config)
            tool_msgs.append(tool_message)

    print("tool_msgs: \n", tool_msgs)
    print("-"*100)
    return fewshot_search_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)


# 체인 실행
query = "스테이크 메뉴가 있나요? 스테이크와 어울리는 와인을 추천해주세요."
response = restaurant_menu_chain.invoke(query)

# 응답 출력
pprint(response.content)

search_menu: 
{'name': 'search_menu', 'args': {'query': '스테이크'}, 'id': 'call_17Aefvd8n1LZjJ2rtPcfimMW', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
search_wine: 
{'name': 'search_wine', 'args': {'query': '스테이크와 어울리는 와인'}, 'id': 'call_udEngtlRb1oxsTDPVznWwTPv', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
tool_msgs: 
 [ToolMessage(content="[Document(id='55420428-0f94-4f82-9383-4e2e3c4cea62', metadata={'menu_number': 1, 'source': './data/restaurant_menu.txt', 'menu_name': '시그니처 스테이크'}, page_content='1. 시그니처 스테이크\\n   • 가격: ₩35,000\\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.'), Document(id='02b0713d-fff7-48fd-ba37-b820c174fffd', metadata={'menu_number': 8, 'source': './data/restaurant_m

In [56]:
# 체인 실행
query = "파스타의 유래에 대해서 알고 있나요? 서울 강남의 파스타 맛집을 추천해주세요."
response = restaurant_menu_chain.invoke(query)

# 응답 출력
pprint(response.content)

wiki_summary: 
{'name': 'wiki_summary', 'args': {'query': '파스타의 유래'}, 'id': 'call_hYFXQSy2hUv5AWz2jFV7ei9m', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
search_menu: 
{'name': 'search_menu', 'args': {'query': '강남 파스타 맛집'}, 'id': 'call_ZEf1BZgr3LQxzRovFGzp6lre', 'type': 'tool_call'}
----------------------------------------------------------------------------------------------------
tool_msgs: 
 [ToolMessage(content='오르조(리소니)는 이탈리아의 파스타로, 보리에서 유래된 이름을 가지고 있으며 큰 쌀알 모양이다. 주로 수프와 함께 먹으며, 현재는 박력분으로 만들어진다. 터키에서는 아르파 셰흐리예로 불린다. 파파르델레는 넓고 긴 형태의 이탈리아 파스타로, "게걸스럽게 먹다"는 뜻의 이탈리아어에서 유래했다. 신선한 파파르델레는 약 3cm 너비이며, 주로 육수와 함께 요리된다. 이 파스타는 이탈리아의 축제에서도 중요한 역할을 한다.', name='wiki_summary', tool_call_id='call_hYFXQSy2hUv5AWz2jFV7ei9m'), ToolMessage(content="[Document(id='0d31bb83-6e20-48b1-bd8c-55935cc2a93a', metadata={'menu_name': '해산물 파스타', 'menu_number': 6, 'source': './data/restaurant_menu.txt'}, page_content='6. 해산물 파스타\\n   • 가

## 4. LangChain Agent 사용
- LangChain v1에서는 `create_agent`가 도구 호출 루프와 메시지 상태를 직접 관리합니다.

In [63]:
from textwrap import dedent
from langchain.agents import create_agent

agent_system_prompt = dedent("""
    You are an AI assistant providing restaurant menu information and general food-related knowledge.
    Your main goal is to provide accurate information and effective recommendations to users.

    Key guidelines:
    1. For restaurant menu information, use the search_menu tool. This tool provides details on menu items, including prices, ingredients, and cooking methods.
    2. For general food information, history, and cultural background, use the wiki_summary tool.
    3. For wine recommendations or food and wine pairing information, use the search_wine tool.
    4. If additional web searches are needed or for the most up-to-date information, use the search_web tool.
    5. Provide clear and concise responses based on the search results.
    6. If a question is ambiguous or lacks necessary information, politely ask for clarification.
    7. Always maintain a helpful and professional tone.
    8. When providing menu information, describe price, main ingredients, and distinctive cooking methods in that order.
    9. When making recommendations, briefly explain the reasons.
    10. Maintain a friendly, engaging, and natural conversational style.

    Understand the purpose of each tool and use multiple tools when necessary.
    Always strive to provide the most current and accurate information.
""")

tools = [search_web, wiki_summary, search_wine, search_menu]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=agent_system_prompt,
    debug=True,
)

In [64]:
# Agent 실행
query = "시그니처 스테이크의 가격과 특징은 무엇인가요? 그리고 스테이크와 어울리는 와인 추천도 해주세요."

agent_response = agent.invoke({
    "messages": [
        {"role": "user", "content": query},
    ]
})

[values] {'messages': [HumanMessage(content='시그니처 스테이크의 가격과 특징은 무엇인가요? 그리고 스테이크와 어울리는 와인 추천도 해주세요.', additional_kwargs={}, response_metadata={}, id='a0dca946-055e-467f-8806-2c79d3e351b9')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 471, 'total_tokens': 524, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8516364442', 'id': 'chatcmpl-E1uEMpkYiv45TZJOjPEAJcdQIFDuN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f6607-5c56-7090-a867-96ca23992235-0', tool_calls=[{'name': 'search_menu', 'args': {'query': '시그니처 스테이크'}, 'id': 'call_FlJkWvNorZvjGFS

In [65]:
# create_agent는 전체 메시지 상태를 반환하므로 마지막 메시지에서 최종 답변을 추출
final_answer = agent_response["messages"][-1].content
pprint(final_answer)

('### 시그니처 스테이크\n'
 '- **가격**: ₩35,000\n'
 '- **주요 식재료**: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\n'
 '- **특징**:셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 '
 '보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.\n'
 '\n'
 '### 스테이크와 어울리는 와인 추천\n'
 '1. **그랜지 2016**\n'
 '   - **가격**: ₩950,000\n'
 '   - **주요 품종**: 시라\n'
 '   - **특징**: 블랙베리, 자두, 블랙 올리브의 강렬한 과실향과 유칼립투스, 초콜릿, 가죽의 복합 향이 어우러져 있으며, 풀바디 '
 '특성과 강렬한 타닌이 특징입니다. 스테이크의 풍미를 잘 보완합니다.\n'
 '\n'
 '2. **사시카이아 2018**\n'
 '   - **가격**: ₩420,000\n'
 '   - **주요 품종**: 카베르네 소비뇽, 카베르네 프랑, 메를로\n'
 '   - **특징**: 블랙베리와 카시스의 강렬한 과실향, 허브, 가죽, 스파이스 노트가 어우러져 우아한 타닌과 신선한 산도가 균형을 '
 '이루고 있어 스테이크와 훌륭하게 어울립니다.\n'
 '\n'
 '이 두 가지 와인은 시그니처 스테이크와 잘 어울릴 것으로 생각되며, 각각의 고유한 향과 풍미를 통해 스테이크의 맛을 한층 끌어올려 줄 '
 '것입니다.')


## 5. Gradio 활용

In [ ]:
from typing import Any

import gradio as gr


def answer_invoke(message: str, history: list[dict[str, Any]]) -> str:
    try:
        # Gradio 6의 OpenAI-style 대화 기록에서 최근 한 턴만 전달
        recent_history = [
            {"role": item["role"], "content": item["content"]}
            for item in history[-2:]
            if item.get("role") in {"user", "assistant"}
            and isinstance(item.get("content"), str)
        ]

        response = agent.invoke({
            "messages": [
                *recent_history,
                {"role": "user", "content": message},
            ]
        })

        final_content = response["messages"][-1].content
        return final_content if isinstance(final_content, str) else str(final_content)
    except Exception as e:
        print(f"Error occurred: {e}")
        return "죄송합니다. 응답을 생성하는 동안 오류가 발생했습니다. 다시 시도해 주세요."


example_questions = [
    "시그니처 스테이크의 가격과 특징을 알려주세요.",
    "트러플 리조또와 잘 어울리는 와인을 추천해주세요.",
    "해산물 파스타의 주요 재료는 무엇인가요? 서울 강남 지역에 레스토랑을 추천해주세요.",
    "채식주의자를 위한 메뉴 옵션이 있나요?",
]

demo = gr.ChatInterface(
    fn=answer_invoke,
    title="레스토랑 메뉴 AI 어시스턴트",
    description="메뉴 정보, 추천, 음식 관련 질문에 답변해 드립니다.",
    examples=example_questions,
    theme=gr.themes.Soft(),
)

demo.launch()

In [ ]:
# 데모 종료
demo.close()